In [1]:
import backtrader as bt
import datetime
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
%matplotlib inline


/home/majidi/anaconda3/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


In [2]:
from datetime import datetime
import backtrader as bt

In [7]:
class RsiSignalStrategy(bt.SignalStrategy):
    params = dict(rsi_periods=14, rsi_upper=70, rsi_lower=30, rsi_mid=50)
    
    def __init__(self):
        rsi = bt.indicators.RSI(period=self.p.rsi_periods, upperband=self.p.rsi_upper, lowerband=self.p.rsi_lower)
        bt.talib.RSI(self.data, plotname='TA_RSI')
        rsi_signal_long = bt.ind.CrossUp(rsi, self.p.rsi_lower, plot=False)
        self.signal_add(bt.SIGNAL_LONG, rsi_signal_long)
        self.signal_add(bt.SIGNAL_LONGEXIT, -(rsi > self.p.rsi_mid))
        rsi_signal_short = -bt.ind.CrossDown(rsi, self.p.rsi_upper, plot=False)
        self.signal_add(bt.SIGNAL_SHORT, rsi_signal_short)
        self.signal_add(bt.SIGNAL_SHORTEXIT, rsi < self.p.rsi_mid)

    def log(self, txt):
        dt = self.datas[0].datetime.date(0).isoformat()
        print(f'{dt}, {txt}')
        
    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            return
        if order.status in [order.Completed]:
            if order.isbuy():
                self.log(f'BUY EXECUTED --- Price: {order.executed.price:.2f}, Cost: {order.executed.value:.2f}, \
                        Commission: {order.executed.comm:.2f}'
                        )
                self.price = order.executed.price
                self.comm = order.executed.comm
            else:
                self.log(
                    f'SELL EXECUTED --- Price: \
                    {order.executed.price:.2f}, Cost: {order.executed.value:.2f}, \
                    Commission: {order.executed.comm:.2f}'
                    )
        elif order.status in [order.Canceled, order.Margin,order.Rejected]:
            self.log('Order Failed')
            self.order = None
                
    def notify_trade(self, trade):
        if not trade.isclosed:
            return self.log(f'OPERATION RESULT --- Gross: {trade.pnl:.2f}, Net: {trade.pnlcomm:.2f}')
        
    def next_open(self):
        if not self.position:
            if self.buy_signal > 0:
                size = int(self.broker.getcash() / self.datas[0].open)
                self.log(f"""BUY CREATED --- Size: {size}, Cash: {self.broker.getcash():.2f}, Open: {self.data_open[0]}, Close: \
                        {self.data_close[0]}""") 
                self.buy(size=size)
        else:
            if self.sell_signal < 0:
                self.log(f'SELL CREATED --- Size: {self.position.size}') 
                self.sell(size=self.position.size)
    


data = yf.download('META', start='2018-01-01', end='2018-12-31')
data.columns = data.columns.droplevel(level=1)
# Convert for Backtrader
datafeed = bt.feeds.PandasData(dataname=data)


cerebro = bt.Cerebro(stdstats = False)
cerebro.addstrategy(RsiSignalStrategy)
cerebro.adddata(datafeed)
cerebro.broker.setcash(1000.0)
cerebro.broker.setcommission(commission=0.001)
cerebro.addobserver(bt.observers.BuySell)
cerebro.addobserver(bt.observers.Value)

print('Starting Portfolio Value: %.2f' % cerebro.broker.getvalue())
backtest_result = cerebro.run()
print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue())


cerebro.plot(iplot=False, volume=False)


/home/majidi/anaconda3/lib/python3.13/site-packages/yfinance/utils.py:559: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/home/majidi/anaconda3/lib/python3.13/site-packages/yfinance/utils.py:505: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/home/majidi/anaconda3/lib/python3.13/site-packages/yfinance/utils.py:509: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
[*********************100%***********************]  

Starting Portfolio Value: 1000.00
2018-03-27, BUY EXECUTED --- Price: 154.83, Cost: 154.83,                         Commission: 0.15
2018-03-27, OPERATION RESULT --- Gross: 0.00, Net: -0.15
2018-04-18, SELL EXECUTED --- Price:                     165.30, Cost: 154.83,                     Commission: 0.17
2018-05-15, SELL EXECUTED --- Price:                     183.13, Cost: -183.13,                     Commission: 0.18
2018-05-15, OPERATION RESULT --- Gross: 0.00, Net: -0.18
2018-07-05, BUY EXECUTED --- Price: 192.89, Cost: -183.13,                         Commission: 0.19
2018-07-27, SELL EXECUTED --- Price:                     178.16, Cost: -178.16,                     Commission: 0.18
2018-07-27, OPERATION RESULT --- Gross: 0.00, Net: -0.18
2018-07-30, BUY EXECUTED --- Price: 173.64, Cost: -178.16,                         Commission: 0.17
2018-08-03, BUY EXECUTED --- Price: 176.00, Cost: 176.00,                         Commission: 0.18
2018-08-03, OPERATION RESULT --- Gross: 0.00, N


/home/majidi/anaconda3/lib/python3.13/site-packages/yfinance/multi.py:215: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  data.rename(columns=ctx.isins, inplace=True)
/home/majidi/anaconda3/lib/python3.13/site-packages/yfinance/multi.py:219: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  data.sort_index(level=0, axis=1, inplace=True)


[[<Figure size 640x480 with 16 Axes>]]

In [8]:
cerebro.plot(iplot=False, volume=False)
plt.show()